# Кейс № 2_1 - Пример

### Генерация данных

Загрузка массива происходит через методы numpy. Генерируем искусственный датасет. Он сохраняется в файл .cvc, для дальнешей работы с ним.

In [1]:
import numpy as np

np.random.seed(7)

n = 120

# ID курса (3 курса)
course_id = np.random.randint(1, 4, size=n)

# День (1–15)
day = np.random.randint(1, 16, size=n)

# Просмотры страницы курса
views = np.random.randint(500, 3000, size=n)

# Записи (примерно 8% от просмотров)
enrollments = (views * 0.08).astype(int)

# Затраты на продвижение (50–150 руб на запись)
costs = enrollments * np.random.uniform(50, 150, size=n)

# Завершения курса (60% от записавшихся)
completions = (enrollments * 0.6).astype(int)

# Объединяем в таблицу NumPy
data = np.column_stack((
    course_id,
    day,
    views,
    enrollments,
    costs,
    completions
))

np.savetxt(
    'courses.csv',
    data,
    delimiter=',',
    header='CourseID,Day,Views,Enrollments,Costs,Completions',
    comments='',
    fmt='%.2f'
)

print("Файл courses.csv создан")


Файл courses.csv создан


Загружаем данные из файла

In [2]:
loaded = np.loadtxt(
    'courses.csv',
    delimiter=',',
    skiprows=1
)

print("Первые 5 строк:")
print(loaded[:5])
print("Форма:", loaded.shape)

Первые 5 строк:
[[1.000000e+00 5.000000e+00 6.760000e+02 5.400000e+01 4.165580e+03
  3.200000e+01]
 [2.000000e+00 2.000000e+00 1.951000e+03 1.560000e+02 2.168740e+04
  9.300000e+01]
 [3.000000e+00 1.400000e+01 1.417000e+03 1.130000e+02 7.949650e+03
  6.700000e+01]
 [1.000000e+00 7.000000e+00 2.692000e+03 2.150000e+02 1.700887e+04
  1.290000e+02]
 [2.000000e+00 1.100000e+01 2.525000e+03 2.020000e+02 2.115405e+04
  1.210000e+02]]
Форма: (120, 6)


Выделение стобцов

In [3]:
course_id = data[:, 0]
views = data[:, 2]
enrollments = data[:, 3]
costs = data[:, 4]
completions = data[:, 5]

### Векторизация

Векторизация — выполнение операций сразу над всем массивом.

CR = (Enrollments / Views) * 100
CPC = Cost / Completions

NumPy применяет операцию ко всем элементам сразу.

Преимущества:

- меньше кода
- быстрее выполнение
- меньше ошибок

На выходе мы получим **новый массив**, в котором будут храниться
рассчитанные метрики для каждой строки таблицы.

То есть:

- для каждого курса (или дня)
- будет своё значение CR
- и своё значение CPC

Мы не храним одну формулу — мы получаем результат её применения
ко всему набору данных.

In [4]:
cr = np.where(
    views > 0,
    enrollments / views * 100,
    np.nan
)

cpc = np.where(
    completions > 0,
    costs / completions,
    np.nan
)


Расчет статистики

In [5]:
print("CR среднее:", {np.nanmean(cr)})
print("CR мин:", np.nanmin(cr))
print("CR макс:", np.nanmax(cr))

print("CPC среднее:", np.nanmean(cpc))
print("CPC мин:", np.nanmin(cpc))
print("CPC макс:", np.nanmax(cpc))

CR среднее: {np.float64(7.966131445725802)}
CR мин: 7.8277886497064575
CR макс: 8.0
CPC среднее: 172.54416762943922
CPC мин: 84.18957926575406
CPC макс: 249.34747604962826


In [12]:
unique_courses = np.unique(course_id)

total_cost = []
total_comp = []

for cid in unique_courses:
    mask = course_id == cid
    total_cost.append(costs[mask].sum())
    total_comp.append(completions[mask].sum())

total_cost = np.array(total_cost)
total_comp = np.array(total_comp)

course_eff = np.where(
    total_comp > 0,
    total_cost / total_comp,
    np.nan
)

top2 = np.argsort(course_eff)[:2]

print("Топ-2 курса по стоимости завершения:")
for i in top2:
    print("Курс", unique_courses[i],
          "Стоимость завершения:", course_eff[i])

Топ-2 курса по стоимости завершения:
Курс 3.0 Стоимость завершения: 165.33292520450902
Курс 2.0 Стоимость завершения: 175.77003237584728
